# Lecture 7 — Entanglement and the Area Law

---

## Overview

Entanglement is one of the defining features of quantum mechanics with no classical counterpart. In many-body physics, it is not just a curiosity — it is a diagnostic tool. The amount of entanglement in the ground state encodes information about its phase: in gapped phases it obeys the **area law** (saturation), while at a critical point it grows logarithmically — a fingerprint that identifies the universality class through the **central charge** $c$ of the underlying CFT.

This lecture connects quantum phase transitions to the mathematics that makes tensor networks possible.

---

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})


def build_tfim(N, J=1.0, Gamma=1.0):
    dim = 2**N
    rows, cols, data = [], [], []
    for state in range(dim):
        diag = 0.0
        for i in range(N - 1):
            sz_i = 1 if (state >> i) & 1 else -1
            sz_j = 1 if (state >> (i + 1)) & 1 else -1
            diag -= J * sz_i * sz_j
        if diag != 0.0:
            rows.append(state); cols.append(state); data.append(diag)
        for i in range(N):
            rows.append(state); cols.append(state ^ (1 << i)); data.append(-Gamma)
    return sp.csr_matrix((data, (rows, cols)), shape=(dim, dim))

## 1. Bipartite Entanglement

Divide a chain of $N$ spins into subsystem $A$ (first $\ell$ sites) and subsystem $B$ (remaining $N-\ell$ sites). A state is a **product state** if $|E_0\rangle = |\psi_A\rangle \otimes |\psi_B\rangle$ — zero entanglement. Otherwise the subsystems are entangled: measuring a spin in $A$ affects probabilities in $B$.

---

## 2. The Reduced Density Matrix

The state of subsystem $A$ alone is characterised by the **reduced density matrix**:

$$\rho_A = \text{Tr}_B |E_0\rangle\langle E_0|$$

A rank-1 $\rho_A$ means no entanglement; multiple non-zero eigenvalues signal entanglement.

---

In [ ]:
def reduced_density_matrix(psi, ell, N):
    """
    Compute the reduced density matrix rho_A for a bipartition A=[0..ell-1], B=[ell..N-1].
    Reshapes the wavefunction as a (2^ell x 2^(N-ell)) matrix and computes psi @ psi^dag.
    """
    dim_A = 2**ell
    dim_B = 2**(N - ell)
    psi_mat = psi.reshape(dim_A, dim_B)
    return psi_mat @ psi_mat.conj().T


def entanglement_entropy(psi, ell, N):
    """Von Neumann entanglement entropy S_A for a cut at site ell."""
    rho_A = reduced_density_matrix(psi, ell, N)
    eigvals = np.linalg.eigvalsh(rho_A)
    eigvals = eigvals[eigvals > 1e-15]  # avoid log(0)
    return -np.sum(eigvals * np.log(eigvals))


# Example: entanglement in the two limits
N = 8

# Pure transverse field (product state)
H_dis = build_tfim(N, J=1.0, Gamma=10.0)
_, evecs_dis = eigsh(H_dis, k=1, which='SA')
psi_dis = evecs_dis[:, 0]

# Critical point
H_crit = build_tfim(N, J=1.0, Gamma=1.0)
_, evecs_crit = eigsh(H_crit, k=1, which='SA')
psi_crit = evecs_crit[:, 0]

print("Entanglement entropy S_A at cut ℓ=N/2:")
print(f"  Disordered phase (Γ/J=10): S = {entanglement_entropy(psi_dis, N//2, N):.4f}")
print(f"  Critical point  (Γ/J=1):  S = {entanglement_entropy(psi_crit, N//2, N):.4f}")

## 3. The Schmidt Decomposition

Any pure bipartite state can be written as $|E_0\rangle = \sum_\alpha \lambda_\alpha |\alpha\rangle_A |\alpha\rangle_B$. The Schmidt coefficients $\lambda_\alpha$ are the singular values of the reshaped coefficient matrix:

$$C = U \Sigma V^\dagger$$

This SVD structure is the foundation of matrix product states.

---

In [ ]:
def schmidt_spectrum(psi, ell, N):
    """Return Schmidt coefficients (singular values) for a cut at site ell."""
    dim_A, dim_B = 2**ell, 2**(N - ell)
    psi_mat = psi.reshape(dim_A, dim_B)
    _, s, _ = np.linalg.svd(psi_mat, full_matrices=False)
    return s


N = 12
ell = N // 2
phases = [(0.3, 'Ordered', 'tab:blue'), (1.0, 'Critical', 'tab:red'), (3.0, 'Disordered', 'tab:green')]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for g, label, color in phases:
    H = build_tfim(N, J=1.0, Gamma=g)
    _, evecs = eigsh(H, k=1, which='SA')
    s = schmidt_spectrum(evecs[:, 0], ell, N)
    axes[0].plot(range(1, len(s)+1), s**2, 'o-', color=color, label=label, markersize=4)
    axes[1].semilogy(range(1, len(s)+1), s**2 + 1e-15, 'o-', color=color, label=label, markersize=4)

for ax in axes:
    ax.set_xlabel(r'Schmidt index $\alpha$')
    ax.set_ylabel(r'$\lambda_\alpha^2$')
    ax.legend(fontsize=9)

axes[0].set_title('Schmidt spectrum (linear)')
axes[1].set_title('Schmidt spectrum (log scale)')
plt.tight_layout()
plt.show()

## 4. Entanglement Entropy

The von Neumann entanglement entropy:

$$S_A = -\text{Tr}(\rho_A \log \rho_A) = -\sum_\alpha \lambda_\alpha^2 \log \lambda_\alpha^2$$

Properties: $S_A = 0$ for product states; $S_A = \log\chi$ for maximally entangled states; $S_A = S_B$.

---

In [ ]:
# Entanglement entropy profile S(ell) for a fixed system size
N = 16
ells = range(1, N)

fig, ax = plt.subplots(figsize=(8, 5))

for g, label, color in phases:
    H = build_tfim(N, J=1.0, Gamma=g)
    _, evecs = eigsh(H, k=1, which='SA')
    psi0 = evecs[:, 0]
    S = [entanglement_entropy(psi0, ell, N) for ell in ells]
    ax.plot(ells, S, 'o-', color=color, label=label, markersize=4)

ax.set_xlabel(r'Subsystem size $\ell$')
ax.set_ylabel(r'$S_A(\ell)$')
ax.set_title(f'Entanglement entropy profile, N={N}')
ax.legend()
plt.tight_layout()
plt.show()

## 5. The Area Law

For ground states of **gapped** local Hamiltonians, $S_A$ saturates to a constant in 1D — the **area law**. Intuitively: in a gapped phase, correlations decay exponentially. Only spins near the boundary between $A$ and $B$ are significantly entangled.

The area law has a profound consequence: a state satisfying it can be efficiently represented as an MPS with a bond dimension $\chi$ that is independent of $N$. This is why DMRG works so well for gapped 1D systems.

---

## 6. Critical Systems: Logarithmic Scaling

At a quantum critical point, the area law is violated. For a 1D critical system described by a CFT (open boundary conditions):

$$S_A(\ell) = \frac{c}{6} \log\!\left(\frac{2N}{\pi} \sin\frac{\pi \ell}{N}\right) + \text{const}$$

This Calabrese-Cardy (2004) formula connects a local numerical measurement to the universal central charge $c$.

---

## 7. The Central Charge

For the TFIM at $\Gamma = J$, the system is described by the Ising CFT with $c = 1/2$. We therefore expect:

$$S_A(\ell) = \frac{1}{6} \log\!\left(\frac{2N}{\pi}\sin\frac{\pi\ell}{N}\right) + \text{const}$$

Fitting this formula to ED data is one of the cleanest ways to identify the universality class of a quantum critical point.

---

In [ ]:
from scipy.optimize import curve_fit

N = 20
H_crit = build_tfim(N, J=1.0, Gamma=1.0)
_, evecs_crit = eigsh(H_crit, k=1, which='SA')
psi_crit = evecs_crit[:, 0]

ells = np.arange(1, N)
S_crit = np.array([entanglement_entropy(psi_crit, int(ell), N) for ell in ells])

# Fit to Calabrese-Cardy formula (OBC): S = (c/6) * log(2N/pi * sin(pi*ell/N)) + const
x_fit = np.log(2*N/np.pi * np.sin(np.pi * ells / N))

def cc_formula(x, c_over_6, const):
    return c_over_6 * x + const

popt, _ = curve_fit(cc_formula, x_fit, S_crit)
c_fitted = popt[0] * 6

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x_fit, S_crit, 'o', label='ED data', markersize=5)
ax.plot(x_fit, cc_formula(x_fit, *popt), '-', label=f'CC fit: $c={c_fitted:.3f}$ (expected 0.5)')
ax.set_xlabel(r'$\log\!\left(\frac{2N}{\pi}\sin\frac{\pi\ell}{N}\right)$')
ax.set_ylabel(r'$S_A(\ell)$')
ax.set_title(f'Central charge extraction, N={N}')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Fitted central charge: c = {c_fitted:.4f} (expected c = 0.5 for Ising CFT)")

## 8. Entanglement across the Phase Diagram

The three-way distinction — two area-law phases separated by a critical point with logarithmic entanglement — is a general feature of 1D quantum systems:

- **Ordered phase** ($\Gamma < \Gamma_c$): $S_A$ saturates — area law.
- **Critical point** ($\Gamma = \Gamma_c$): $S_A \sim \frac{1}{6}\log\ell$ — logarithmic growth.
- **Disordered phase** ($\Gamma > \Gamma_c$): $S_A$ saturates again, but to a smaller constant.

For a state satisfying the area law, the Schmidt decomposition has only $\chi = e^{S_{\max}}$ significant singular values. Truncating to these $\chi$ values gives a faithful representation as a **Matrix Product State** — the subject of lecture 08.

---

In [ ]:
# S(ell=N/2) as a function of Gamma/J — area law vs logarithmic
N = 18
gammas_ee = np.linspace(0.2, 2.5, 50)
S_half = []

for g in gammas_ee:
    H = build_tfim(N, J=1.0, Gamma=g)
    _, evecs = eigsh(H, k=1, which='SA')
    S_half.append(entanglement_entropy(evecs[:, 0], N//2, N))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(gammas_ee, S_half, 'b-o', markersize=4)
ax.axvline(1.0, color='red', linestyle='--', label=r'$\Gamma_c=J$')
ax.set_xlabel(r'$\Gamma/J$')
ax.set_ylabel(r'$S_A(\ell=N/2)$')
ax.set_title(f'Half-chain entanglement entropy, N={N}')
ax.annotate('Area law', xy=(0.4, S_half[5]), fontsize=10, color='tab:blue')
ax.annotate('Area law', xy=(1.7, S_half[-10]), fontsize=10, color='tab:blue')
ax.annotate('Log scaling\n(CFT peak)', xy=(1.05, max(S_half)*0.97), fontsize=10, color='tab:red')
ax.legend()
plt.tight_layout()
plt.show()